In [ ]:
from google.colab import drive, userdata
from getpass import getpass
DRIVE_CONNECTED = False
DRIVE_MODE = 'local-temporary'
DRIVE_DATA = pathlib.Path('/content/AuraAI-data')
try:
    drive.mount('/content/drive', force_remount=True)
    DRIVE_DATA = pathlib.Path('/content/drive/MyDrive/AuraAI/data')
    DRIVE_CONNECTED = True
    DRIVE_MODE = 'mounted-filesystem'
    print('Google Drive 파일 시스템 연결 완료')
except Exception as mount_error:
    print('Drive 파일 시스템 연결 실패:', type(mount_error).__name__, str(mount_error)[:180])
    try:
        from google.colab import auth
        auth.authenticate_user()
        import google.auth, hashlib, mimetypes
        from googleapiclient.discovery import build
        from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
        credentials, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive'])
        drive_service = build('drive', 'v3', credentials=credentials, cache_discovery=False)
        def drive_find_folder(name, parent_id):
            safe = name.replace("'", "\\\\'")
            q = f"name='{safe}' and mimeType='application/vnd.google-apps.folder' and '{parent_id}' in parents and trashed=false"
            found = drive_service.files().list(q=q, pageSize=100, fields='files(id,name)').execute().get('files', [])
            if found: return found[0]['id']
            return drive_service.files().create(body={'name':name,'mimeType':'application/vnd.google-apps.folder','parents':[parent_id]},fields='id').execute()['id']
        root_folder_id = drive_find_folder('AuraAI', 'root')
        data_folder_id = drive_find_folder('data', root_folder_id)
        def drive_list(parent_id):
            items, token = [], None
            while True:
                result = drive_service.files().list(q=f"'{parent_id}' in parents and trashed=false", pageSize=1000, pageToken=token, fields='nextPageToken,files(id,name,mimeType,md5Checksum,size)').execute()
                items.extend(result.get('files', [])); token=result.get('nextPageToken')
                if not token: return items
        def drive_download_tree(folder_id, local_dir):
            local_dir.mkdir(parents=True, exist_ok=True)
            for item in drive_list(folder_id):
                target=local_dir/item['name']
                if item['mimeType']=='application/vnd.google-apps.folder':
                    drive_download_tree(item['id'], target)
                elif not target.exists():
                    from googleapiclient.http import MediaIoBaseDownload
                    import io
                    request=drive_service.files().get_media(fileId=item['id']); stream=io.BytesIO(); downloader=MediaIoBaseDownload(stream,request)
                    done=False
                    while not done: _,done=downloader.next_chunk()
                    target.parent.mkdir(parents=True,exist_ok=True); target.write_bytes(stream.getvalue())
        def drive_upload_tree(local_dir, folder_id):
            remote={item['name']:item for item in drive_list(folder_id)}
            for local in local_dir.iterdir():
                if local.is_dir():
                    entry=remote.get(local.name)
                    child_id=entry['id'] if entry and entry['mimeType']=='application/vnd.google-apps.folder' else drive_service.files().create(body={'name':local.name,'mimeType':'application/vnd.google-apps.folder','parents':[folder_id]},fields='id').execute()['id']
                    drive_upload_tree(local,child_id)
                elif local.is_file() and not local.name.endswith('.tmp'):
                    digest=hashlib.md5(local.read_bytes()).hexdigest(); entry=remote.get(local.name)
                    if entry and entry.get('md5Checksum')==digest: continue
                    media=MediaFileUpload(str(local),mimetype=mimetypes.guess_type(local.name)[0] or 'application/octet-stream',resumable=True)
                    if entry: drive_service.files().update(fileId=entry['id'],media_body=media).execute()
                    else: drive_service.files().create(body={'name':local.name,'parents':[folder_id]},media_body=media).execute()
        DRIVE_CONNECTED=True
        DRIVE_MODE='drive-api-sync'
        print('Drive API 인증 성공. 파일 시스템 마운트 대신 Drive API 동기화를 사용합니다.')
    except Exception as api_error:
        print('Drive API 인증도 실패:', type(api_error).__name__, str(api_error)[:180])
        DRIVE_CONNECTED=False
DRIVE_DATA.mkdir(parents=True, exist_ok=True)
if DRIVE_MODE=='drive-api-sync':
    drive_download_tree(data_folder_id, DRIVE_DATA)
REPO_DATA = pathlib.Path(REPO) / 'data'
for name in ['text', 'image_text', 'audio_text', 'checkpoints', 'tokenizer']:
    src, dst = REPO_DATA / name, DRIVE_DATA / name
    dst.mkdir(parents=True, exist_ok=True)
    if src.is_symlink(): src.unlink()
    elif src.exists():
        for item in src.iterdir():
            target = dst / item.name
            if not target.exists(): shutil.copytree(item, target) if item.is_dir() else shutil.copy2(item, target)
        shutil.rmtree(src)
    if not src.exists(): src.symlink_to(dst, target_is_directory=True)
if DRIVE_MODE=='drive-api-sync':
    drive_upload_tree(DRIVE_DATA, data_folder_id)
    def drive_sync_worker():
        while True:
            time.sleep(180)
            try: drive_upload_tree(DRIVE_DATA, data_folder_id)
            except Exception as sync_error: print('Drive API sync retry:', type(sync_error).__name__, str(sync_error)[:120])
    threading.Thread(target=drive_sync_worker, daemon=True).start()
os.environ['AURA_DEVICE'] = 'cuda' if torch.cuda.is_available() else 'cpu'
os.environ['AURA_GPU_MEMORY_FRACTION'] = '0.75'
os.environ['AURA_CPU_THREADS'] = '4'
def read_colab_secret(name, prompt_label):
    value = ""
    try:
        value = userdata.get(name) or ""
    except Exception as secret_error:
        print(name + " Colab Secret read:", type(secret_error).__name__)
    if not value:
        value = getpass(prompt_label + " (입력값은 표시되지 않습니다): ")
    if not value:
        raise RuntimeError(name + " 값이 비어 있습니다.")
    return value

AURA_GATEWAY_SECRET = read_colab_secret(
    "AURA_GATEWAY_SECRET",
    "Vercel의 AURA_GATEWAY_SECRET과 동일한 게이트웨이 비밀값을 입력하세요"
)
AURA_CLOUDFLARED_TOKEN = read_colab_secret(
    "AURA_CLOUDFLARED_TOKEN",
    "Cloudflare의 aura-ai-api Tunnel 토큰을 입력하세요"
)
os.environ["AURA_GATEWAY_SECRET"] = AURA_GATEWAY_SECRET

TOKEN_FILE = pathlib.Path("/content/.cloudflared-token")
TOKEN_FILE.write_text(AURA_CLOUDFLARED_TOKEN, encoding="utf-8")
os.chmod(TOKEN_FILE, 0o600)
del AURA_CLOUDFLARED_TOKEN

print("Gateway secret loaded (hidden). Device:", os.environ["AURA_DEVICE"])
print("Fixed inference hostname: https://inference.aura.koharu.live")
print("Drive persistence mode:", DRIVE_MODE)


## 자체 학습 데이터

외부 사전학습 모델이나 외부 공개 데이터셋을 사용하지 않습니다. Aura 자체 Transformer와 토크나이저를 처음부터 학습합니다. 학습은 기존 체크포인트에서 이어서 진행됩니다.


## 자체 데이터 기반 학습\n외부 모델·Hugging Face·외부 학습 데이터셋을 다운로드하지 않습니다. Aura 자체 토크나이저와 Transformer를 초기화하고 프로젝트의 자체 데이터만 사용합니다. 첫 체크포인트가 저장되기 전에는 채팅 추론을 시작할 수 없습니다.\n

In [ ]:
from aura.training.runner import RUNNER
from aura.config import TRAINING
TRAINING.update({
    "batch_size": 2 if torch.cuda.is_available() else 1,
    "sequence_length": 512,
    "checkpoint_every": 100,
    "sleep_seconds_per_step": 0.0,
    "max_steps_per_session": 1_000_000,
    "learning_rate": 1.5e-4,
    "warmup_steps": 1000,
    "ram_budget_gb": 12.0 if torch.cuda.is_available() else 8.0,
})
print("Training profile:", "Colab GPU" if torch.cuda.is_available() else "CPU fallback",
      "| batch:", TRAINING["batch_size"], "| sequence:", TRAINING["sequence_length"],
      "| checkpoint interval:", TRAINING["checkpoint_every"])
has_data = any((DRIVE_DATA / 'text').glob('*.txt')) or (DRIVE_DATA / 'image_text' / 'pairs.jsonl').exists() or (DRIVE_DATA / 'audio_text' / 'pairs.jsonl').exists()
if has_data:
    print("Resuming/starting extended training:", RUNNER.start())
else:
    raise RuntimeError("학습 데이터가 없습니다. Drive의 AuraAI/data/text에 UTF-8 TXT 파일을 넣고 다시 실행하세요.")


In [ ]:
import subprocess, threading, requests, pathlib, urllib.request, os, time, sys
API_LOG = open('/content/aura-api.log', 'a', buffering=1)
TUNNEL_LOG = open('/content/aura-tunnel.log', 'a', buffering=1)
PUBLIC_URL = 'https://inference.aura.koharu.live'
TOKEN_FILE = pathlib.Path('/content/.cloudflared-token')
CLOUDFLARED = '/usr/local/bin/cloudflared'
if not pathlib.Path(CLOUDFLARED).exists():
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', CLOUDFLARED)
    os.chmod(CLOUDFLARED, 0o755)
process_lock = threading.Lock()
processes = {'api': None, 'tunnel': None}
def start_api():
    p = processes['api']
    if p is None or p.poll() is not None:
        processes['api'] = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'api.main:app', '--host', '127.0.0.1', '--port', '8000'], cwd=REPO, env=os.environ.copy(), stdout=API_LOG, stderr=subprocess.STDOUT)
        print('Aura API process restarted.')
def wait_api():
    for _ in range(60):
        try:
            r = requests.get('http://127.0.0.1:8000/api/health', timeout=2)
            if r.ok: return True
        except requests.RequestException: pass
        time.sleep(2)
    return False
def start_tunnel():
    p = processes['tunnel']
    if p is None or p.poll() is not None:
        if not TOKEN_FILE.exists():
            raise RuntimeError('고정 터널 토큰 파일이 없습니다. AURA_CLOUDFLARED_TOKEN Secret을 확인하세요.')
        processes['tunnel'] = subprocess.Popen(
            [CLOUDFLARED, 'tunnel', '--no-autoupdate', 'run', '--token-file', str(TOKEN_FILE)],
            env=os.environ.copy(), stdout=TUNNEL_LOG, stderr=subprocess.STDOUT
        )
        print('Named Tunnel connector started for fixed hostname.')
def wait_public_endpoint(timeout=180):
    headers = {"X-Aura-Gateway-Key": os.environ["AURA_GATEWAY_SECRET"]}
    for _ in range(timeout // 3):
        try:
            response = requests.get(PUBLIC_URL + "/api/model/status", headers=headers, timeout=5)
            if response.ok:
                return True
        except requests.RequestException:
            pass
        time.sleep(3)
    return False

def watchdog():
    while True:
        with process_lock:
            start_api()
            if wait_api(): start_tunnel()
        time.sleep(15)
with process_lock:
    start_api()
if not wait_api(): raise RuntimeError('Aura API가 시작되지 않았습니다. /content/aura-api.log 확인')
with process_lock: start_tunnel()
threading.Thread(target=watchdog, daemon=True).start()
if not wait_public_endpoint():
    print("고정 주소에 아직 연결되지 않았습니다. /content/aura-tunnel.log 확인:", PUBLIC_URL)
else:
    print("Fixed inference endpoint is online:", PUBLIC_URL)
print("Vercel AURA_INFERENCE_URL is configured to this fixed hostname.")
print("Local health:", requests.get('http://127.0.0.1:8000/api/health', timeout=5).text)



## 고정 주소와 재접속
- 고정 주소: https://inference.aura.koharu.live
- aura-ai-api 이름의 원격 관리 터널에 Colab을 커넥터로 연결합니다. Colab 런타임이 다시 시작되어도 hostname은 바뀌지 않습니다.
- Cloudflare 대시보드의 aura-ai-api 터널에 inference.aura.koharu.live → http://127.0.0.1:8000 공개 호스트 경로가 사전 설정되어 있습니다.
- Colab Secrets에 AURA_GATEWAY_SECRET과 AURA_CLOUDFLARED_TOKEN 두 항목을 추가해야 합니다. 첫 번째 값은 Vercel의 게이트웨이 비밀값과 일치해야 하며, 두 번째 값은 해당 터널의 커넥터 토큰입니다.
- 노트북은 Secret Store에서 값을 직접 읽고, 누락된 경우 화면에 노출되지 않는 입력을 요청합니다. 터널 토큰은 권한 제한 파일로 저장하며 노트북 코드에 포함하지 않습니다.
- Colab 런타임 자체가 종료되면 모델 프로세스도 종료됩니다. 새 런타임에서 런타임 → 모두 실행을 실행하면 Drive 체크포인트에서 학습을 이어가고 같은 고정 주소로 재연결합니다.
- 학습 상태: requests.get('http://127.0.0.1:8000/api/training/status').json()
- 추론 상태: requests.get(PUBLIC_URL + '/api/model/status', headers={'X-Aura-Gateway-Key': os.environ['AURA_GATEWAY_SECRET']}).json()
